In [ ]:
%load_ext autoreload
%autoreload 2

# Must be set before numpy/scipy/statsmodels are imported (BLAS reads these at
# init). Multi-threaded MKL on this machine crashes the whole process outright
# (no exception, no crash dump) on any sizeable matrix op -- e.g. a single
# np.dot on a ~7.8M-row array, which statsmodels' Logit.predict() calls on
# every optimizer iteration. Forcing single-threaded BLAS fixed it completely
# (verified: the same np.dot that reliably killed the kernel went from
# crashing to running in 0.46s). Single-threaded is somewhat slower per BLAS
# call, but the actual matrix sizes here are small enough (~7.8M x ~130) that
# it's still sub-second either way.
import os
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'

import pipeline
import networkx as nx
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
from collections import defaultdict
import geopandas as gpd
from tqdm.auto import tqdm
import math
from scipy.stats import wasserstein_distance
from matplotlib.colors import to_rgb
import gc

import plotly.io as pio
pio.renderers.default = "notebook"

seed = 21
interval = 6
agg = True

if agg:
    if interval == 6:
        fpath_pos = Path('data/agg/DEP_NET_UNDIRECT_GEOTAX11_14460_20190101_20191231_Tc6_Ts1.csv.gz')
        fpath_neg = Path('data/agg/DEP_NET_UNDIRECT_NEG_GEOTAX11_14460_20190101_20191231_Tc6_Ts1.csv.gz')
        save_dir = Path(f'data/agg/tests/seed{seed}')
    elif interval == 24:
        fpath = Path('data/agg/geotax_complete_24h.pkl')
        save_dir = Path(f'data/agg/tests/seed{seed}')
    else:
        print("Invalid time range (must be 24 or 6)")
else:
    if interval == 6:
        fpath = Path('data/poi_level/6h/thresholded_networks/0_network.txt')
        save_dir = Path(f'data/poi_level/6h/tests/seed{seed}')
        data = Path('data/poi_level/depnet_undirected.gz')
    elif interval == 24:
        fpath = Path('data/poi_level/24h/0_network_24h.txt')
        save_dir = Path(f'data/poi_level/24h/tests/seed{seed}')
        data = Path('data/poi_level/24h/depnet_undirected_24h.gz')
    else:
        print("Invalid time range (must be 24 or 6)")

save_dir.mkdir(parents=True, exist_ok=True)

def fast_read_csv(fpath, _dtypes):
    headers = pd.read_csv(fpath, nrows=0).columns
    _dtypes = {col: dt for col, dt in _dtypes.items() if col in headers}
    return pd.read_csv(fpath, dtype=_dtypes)

print(f"Interval: {interval} hrs | agg: {agg} | Seed: {seed}")

# Setup

In [ ]:
# set parameters

hyperparameters = {
    'p': 1,
    'q': 1,
    'workers': 6,
    'verbose': True,
    'dim': 128,
    'num_walks': 10,
    'walk_length': 80,
    'window_size': 10,
    'epochs': 4,
}

controlled = True
weight = 'cov'
wgt_log = True
if weight:
    weighted = True
else:
    weighted = False
capweight = 'DEP' if weight == 'dep' else (
    'N_COVISITS' if weight == 'cov' else None)
strength = False
operator = 'avg'

print(f'weight={weight} (weighted={weighted}) | strength={strength}')

In [ ]:
# --- new split if needed ---

print(f"Interval: {interval} hrs | agg={agg} | Seed: {seed} | 'Weight: {weight} | Strength: {strength})'")

split_path = save_dir / f"split_{seed}_{weight}_{controlled}.pkl"
train_path = save_dir / f"train_{seed}_{weight}.txt"

if split_path.is_file():
    print(f"Split already exists for seed {seed} weight {weight} controlled {controlled}. Exiting...")
    sys.exit()

G, train_neg, test_pos, test_neg = pipeline.prepare_data(
    fpath, test_frac=.25, seed=seed, weight=weight, agg=agg, meta=True,
    trainfile=train_path, controlled=controlled)

save_dir.mkdir(parents=True, exist_ok=True)
with open(split_path, 'wb') as f:
    pickle.dump((G, train_neg, test_pos, test_neg), f, pickle.HIGHEST_PROTOCOL)
print(f"Saved split at {split_path}")

Outside data

In [ ]:
# --- read in and save outside data if needed ---

# downcast for better performance
_dtypes = {
    'NODE_A': 'category', 'NODE_B': 'category',
    'N_COVISITS': 'float32', 'DIST_KM_MIN': 'float32',
    'DIST_KM_MEAN': 'float32', 'DIST_KM_MEDIAN': 'float32',
    'DIST_KM_CENTROID': 'float32', 'N_UIDS_A': 'float32',
    'N_POIS_A': 'float32', 'N_VISITS_A': 'float32',
    'N_UIDS_B': 'float32', 'N_POIS_B': 'float32',
    'N_VISITS_B': 'float32', 'DEP': 'float32'
}

df_pos = fast_read_csv(fpath_pos, _dtypes)
df_neg = fast_read_csv(fpath_neg, _dtypes)

# divide into train and test
def train_test_filter(df):
    train = df[df['SPLIT'] == 'TRAIN']
    test = df[df['SPLIT'] == 'TEST']
    return train, test

train_pos, test_pos = train_test_filter(df_pos)
train_neg, test_neg = train_test_filter(df_neg)

split_list = (train_pos, train_neg, test_pos, test_neg)
with open(save_dir / 'split_external_matched.pkl', 'wb') as f:
    pickle.dump(split_list, f)

In [ ]:
# --- write training graph from outside data if needed ---
if weight:
    print(f'wgt_log = {wgt_log}')
    G_train = nx.from_pandas_edgelist(
        train_pos, 'NODE_A', 'NODE_B', edge_attr=capweight)
    # rename to 'weight' for node2vec
    for u, v, data in G_train.edges(data=True):
        wgt_val = data.pop(capweight)
        if wgt_log:
            data['weight'] = np.log1p(wgt_val) if wgt_val > 0 else 0
        else:
            data['weight'] = wgt_val
    train_path = save_dir / f"train_external_matched_w{weight}_log{wgt_log}.txt"
else:
    G_train = nx.from_pandas_edgelist(train_pos, 'NODE_A', 'NODE_B')
    train_path = save_dir / "train_external_matched_wNone.txt"

if not train_path.is_file():
    with open(train_path, 'w') as f:
        for u, v, d in G_train.edges(data=True):
            f.write(f"{u}\t{v}\t{d.get('weight', 1.0)}\n")
        print(
            f"Wrote training graph: {G_train.number_of_nodes()} nodes, {G_train.number_of_edges()} edges")
else:
    print("Skipped. (File already exists)")

# Run

### Link Prediction

In [ ]:
# --- reusable function ---

def run_sweep(
        feats_list, split_path, train_path, 
        reuse_file=None
    ):
    
    print(
        "Params check:\n"
        "============\n"
        f"controlled:{controlled}\n"
        f"weight:{weight} (weighted={weighted})\n"
        f"wgt_log:{wgt_log}\n"
        f"strength:{strength}\n"
        f"seed:{seed}\n"
        f"operator:{operator}\n"
    )

    green_light = input("Params ok? Y/N")
    if not green_light == "y" or green_light == "Y":
        print("Exiting...")
        sys.exit()

    # bring in split
    with open(save_dir / split_path, 'rb') as f:
            train_pos, train_neg, test_pos, test_neg = pickle.load(f)

    # default values
    emb_map = None
    emb_store = []
    emb_bool = any('emb' in feats for feats in feats_list)

    if emb_bool:
        # sort into reuse/not
        if reuse_file:
            print("Reusing embeddings...\n")
            with open(save_dir / reuse_file, 'rb') as f:
                prev_results = pickle.load(f)
            emb_map = prev_results['embedding_map']
        else:
            print("Generating new embeddings...\n")
            emb_map = None
            emb_store = []

    # begin loop
    def feats_to_txt(
        feats_list): return '+'.join(feats_list) if isinstance(feats_list, list) else feats_list
    for feats in feats_list:
        feats_id = feats_to_txt(feats)
        print(f'NOW TESTING: {feats_id}\n============')
        if not reuse_file:
            # grab embedding map after first run
            if len(emb_store) == 1:
                emb_map = emb_store[0]
        result = pipeline.run_pipeline(save_dir / train_path,
                            train_pos, train_neg, test_pos, test_neg, features=feats, seed=seed, **hyperparameters,
                            operator=operator, weighted=weighted, agg=agg, embedding_map=emb_map, strength=strength,
                            dyadic_se=True)

        auc = result['link_auc']
        result['link_model'].remove_data()
        result = pd.Series(result)

        # save path handling
        if weight and emb_bool:
            out_file = save_dir / \
                f'matching{controlled}/weight_{weight}/strength{strength}/feats__{feats_id}/{operator}/wgtlog{wgt_log}'
        else:
            if weight and not emb_bool:
                print("\nWeight irrelevant without embeddings. Saving to unweighted folder.")
            out_file = save_dir / \
                f'matching{controlled}/weight_None/strength{strength}/feats__{feats_id}/{operator}'
        if out_file.exists():
            overwrite_gate = input(f"Save path already occupied:\n{out_file}\nOverwrite? Y/N")
            if not overwrite_gate == 'y' or overwrite_gate == 'Y':
                print('\nAll tests completed.')
                print("Exiting...")
                sys.exit()
        out_file.mkdir(parents=True, exist_ok=True)

        with open(out_file / 'auc_score.txt', 'w') as f:
            f.write(f'AUC: {auc:.6f}\n')
            f.write(f'==========\nParams: {hyperparameters}\n')
            print('Score saved.')
        with open(out_file / 'output.pkl', 'wb') as f:
            result.to_pickle(f)
            print('Output saved.\n')
        if not reuse_file:
            if not emb_store:
                emb_store.append(result['embedding_map'])
        del result

        print('\nAll tests completed.')

In [ ]:
# --- single run --- (curr: avg-cosine combo)

print(f"Interval: {interval} hrs | agg: {agg} | Seed: {seed} | 'Weight: {weight} | Strength: {strength}'")

feats = ['emb', 'cosine', 'dist_median']

result = pipeline.run_pipeline(
    save_dir / f"train_external_matched_w{weight}_log{wgt_log}.txt", train_pos, train_neg, 
    test_pos, test_neg, features=feats, seed=seed, **hyperparameters, 
    weighted=weighted, agg=agg, strength=strength, operator='avg'
)

if strength:
    link_auc, str_auc = result['link_auc'], result['str_auc']
    print(f"Link AUC: {link_auc:.4f} | Strength AUC: {str_auc:.4f}")
    result['link_model'].remove_data()
    result['str_model'].remove_data()
    output = pd.Series(result)
else:
    link_auc = result['link_auc']
    print(f"Link AUC: {link_auc:.4f}")
    result['link_model'].remove_data()
    output = pd.Series(result)
    emb_map = result['embedding_map']

# save both score as txt and full output as pickle
feats_id = '__'.join(feats) if isinstance(feats, list) else feats
out_file = save_dir / f'matching{controlled}/weight_{weight}/strength{strength}'
out_file.mkdir(parents=True, exist_ok=True)
with open(out_file / 'auc_scores.txt', 'w') as f:
    f.write(str(link_auc) + '\n')
    if strength:
        f.write(str(str_auc))
    print('Score saved.')
with open(out_file / 'output.pkl', 'wb') as f:
    output.to_pickle(f)
    print('Output saved.')

In [ ]:
# --- sweep --- (curr: weighted dist_mean and dist_median)

split_path = 'split_external_matched.pkl'
if weight:
    train_path = f'train_external_matched_w{weight}_log{wgt_log}.txt'
else:
    train_path = 'train_external_matched_wNone.pkl'

run_sweep([['dist_mean']], split_path, train_path, 
      reuse_file=rf'matching{controlled}\weight_{weight}\strength{strength}\feats__emb+dist_mean\avg\wgtlog{wgt_log}\output.pkl')

### Weight Prediction